###Data Reading JSON

In [0]:
df_json=spark.read.format("json").option("inferSchema",True).option("header",True).option("multiLine",False).load("/Volumes/workspace/databricks_pyspark_tutorial/mydatavolume/drivers.json")

In [0]:
df_json.display()

####using dbutils to see what files inside the volumes

In [0]:
dbutils.fs.ls("/Volumes/workspace/databricks_pyspark_tutorial/mydatavolume")

###Data Reading

In [0]:
df=spark.read.format("csv").option("inferSchema",True).option("header",True).load("/Volumes/workspace/databricks_pyspark_tutorial/mydatavolume/BigMart Sales.csv")

In [0]:
df.display()

In dataframe if we didn't do inferschema as true , then manually write the schema .This ca be done in 2 methods ,
DDL and StrucType()

Schema Definition

In [0]:
df.printSchema()

###DDL SCHEMA

In [0]:
my_ddl_schema='''
Item_identifier string,
Item_Weight string,
Item_Fat_Content string,
Item_Visibility double,
Item_Type string,
Item_MRP double,
Outlet_Identifier string,
Outlet_Establishment_Year integer,
Outlet_Size string,
Outlet_Location_Type string,
Outlet_Type string,
Item_Outlet_Sales double'''

In [0]:
df=spark.read.format('csv').option('header',True).schema(my_ddl_schema).load("/Volumes/workspace/databricks_pyspark_tutorial/mydatavolume/BigMart Sales.csv")


In [0]:
df.display()

In [0]:
df.printSchema()


###StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_struct_schema=StructType([
                             StructField('Item_Identifier',StringType(),True)  ,
                             StructField('Item_Weight',StringType(),True), StructField('Item_Fat_Content',StringType(),True), StructField('Item_Visibility',StringType(),True), StructField('Item_MRP',StringType(),True), StructField('Outlet_Identifier',StringType(),True), StructField('Outlet_Establishment_Year',StringType(),True), StructField('Outlet_Size',StringType(),True), StructField('Outlet_Location_Type',StringType(),True), StructField('Outlet_Type',StringType(),True), StructField('Item_Outlet_Sales',StringType(),True)
])

In [0]:
df=spark.read.format("csv").schema(my_struct_schema).option('header',True).load("/Volumes/workspace/databricks_pyspark_tutorial/mydatavolume/BigMart Sales.csv")


In [0]:
df.printSchema()

###Running the normal schema generated through inferSchema to do further transformations

In [0]:
df.display()

###Transformations

###SELECT

In [0]:
df.select('Item_Identifier','Item_Weight','Item_Fat_Content').display()

In [0]:
df.select(col('Item_Identifier'),col('Item_Weight'),col('Item_Fat_Content')).display()

###ALIAS 
cannot use Alias without an column object

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).display()

In [0]:
df.display()

###FILTER/WHERE

####Scenario 1
In Item_Fat_Content pick only Regular

In [0]:
df.filter(col('Item_Fat_Content')=='Regular').display()

####Scenario 2
Filtering 2 cols at a time

In [0]:
df.filter( (col('Item_Type')=='Soft Drinks')& (col('Item_Weight')<10) ).display()

####Scenario 3
fetch the data with Tier1 or Tier2 and Outlet size is Null

In [0]:
df.filter((col('Outlet_Location_Type').isin('Tier 1','Tier 2')& (col("Outlet_Size").isNull()))).display()

###withColumnRenamed

In [0]:
df.withColumnRenamed('Item_Weight','Item_wt').display()

###withColumn
creating the new column and helps modifying the column

####Scenario 1
Creating new column
if you want to make constant value in the col , you have to function called lit()
Syntax of withColumn:
withColumn('New/Existing column name')  (important)

In [0]:
df=df.withColumn('flag',lit('new'))

In [0]:
df.display()

In [0]:
df.withColumn('multiply',col('Item_Weight')*col('Item_MRP')).display()

####Scenario 2
Modifying the existing columns

In [0]:
df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),'Regular',"Reg"))\
    .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),'Low Fat','LF')).display()

###Type Casting

In [0]:
df=df.withColumn('Item_Weight',col('Item_Weight').cast(StringType()))

In [0]:
df.printSchema()

###Sort/orderBy

####Scene 1
Based on a particular column we sorting the dataframe in descending order or ascending order.

In [0]:
df.sort(col('Item_Weight').desc()).display()

####Scene2


In [0]:
df.sort(col('Item_Visibility').asc()).display()

####Scene3
Perform sorting based on multiple columns
True :1 ,False:0

In [0]:
df.sort(["Item_Weight","Item_Visibility"],ascending=[False,False]).display()

####Scene 4


In [0]:
df.sort(["Item_Weight","Item_Visibility"],ascending=[False,True]).display()

###Limit

In [0]:
df.limit(10).display()

###DROP

####Scene1
Dropping single column

In [0]:
df.drop(col('Item_Visibility')).display()

####Scene2
Dropping Multiple colums

In [0]:
df.drop('Item_Visibility','Item_Type').display()

###DROP_DUPLICATES
If we have duplicate row in dataframe , we can do remove them with this.
if we dont want duplicates in a particular column , we can perform using this.

####Scene1

In [0]:
df.dropDuplicates().display()